# Geometría del Sol y la Tierra desde un lugar lunar

Configura lugar e intervalo en la primera celda y ejecuta de arriba abajo. El Apolo 11
es solo el ejemplo inicial. Los cálculos se comparten con la API Python del proyecto.


In [1]:
from datetime import timedelta
import numpy as np
import pandas as pd
from backend.app.astronomy.lunar_geometry import COLUMNAS_POSICION, calcular_desde_lugar

# CONFIGURACIÓN: modifica este bloque y vuelve a ejecutar las celdas.
LUGAR = {
    "nombre": "Apolo 11",
    "latitud_deg": 0.67416,     # Norte positivo, sur negativo; [-90, 90].
    "longitud_deg": 23.47314,  # Este positivo, oeste negativo; [-180, 360].
    "altura_m": 0.0,           # Metros sobre la superficie de referencia de Horizons.
}
INICIO_UTC = "1969-07-20 20:17:40"  # UTC, no hora local de España.
DURACION = timedelta(0)             # Un instante; para una serie: timedelta(hours=6).
PASO = timedelta(minutes=10)        # Minutos enteros; también admite hours=1, days=1.
FILAS_VISTA = 10                    # Número de filas mostradas, no limita la consulta.
MOSTRAR_CABECERAS = False           # Mostrar los detalles del modelo de Horizons.

# CONSTANTES Y COLUMNAS: definidas una sola vez.
CUERPOS = ("tierra", "sol")
COLUMNAS_ARRAY = ["t_segundos", *COLUMNAS_POSICION]
COLUMNAS_GEOMETRIA = [
    f"{magnitud}_{cuerpo}_{unidad}"
    for cuerpo in CUERPOS
    for magnitud, unidad in (("elevacion", "deg"), ("acimut", "deg"),
                            ("angulo_normal", "deg"), ("distancia", "km"))
]
COLUMNAS_BOOLEANAS = [
    "sol_sobre_horizonte", "tierra_bloquea_sol", "iluminado_sin_terreno",
    "tierra_visible", "conexion_tierra",
]
COLUMNAS_TERRENO = ["tiempo_utc", "t_segundos", *COLUMNAS_GEOMETRIA, *COLUMNAS_BOOLEANAS]

CAMPOS_RESUMEN = {
    "x_km": "{cuerpo}_x_km", "y_km": "{cuerpo}_y_km", "z_km": "{cuerpo}_z_km",
    "distancia_km": "distancia_{cuerpo}_km", "elevacion_deg": "elevacion_{cuerpo}_deg",
    "acimut_deg": "acimut_{cuerpo}_deg", "angulo_normal_deg": "angulo_normal_{cuerpo}_deg",
}


## Módulo compartido con el backend

El cálculo se importa de `backend/app/astronomy/lunar_geometry.py`. La API y este notebook
usan la misma implementación. Ejecuta el notebook desde la raíz del repositorio.


## Consulta y resultados

Esta es la única celda que consulta la API. `tabla` contiene todos los campos y `tabla_terreno`
selecciona los que necesita el cálculo topográfico. Los arrays se extraen de esa misma tabla.


In [2]:
tabla = calcular_desde_lugar(INICIO_UTC, DURACION, PASO, **LUGAR)
tabla_terreno = tabla[COLUMNAS_TERRENO].copy()

datos = tabla[COLUMNAS_ARRAY].to_numpy(dtype=float)
datos_geometria = tabla[["t_segundos", *COLUMNAS_GEOMETRIA]].to_numpy(dtype=float)
tiempos_utc = tabla["tiempo_utc"].to_numpy()
posiciones_tierra_km = tabla[[f"tierra_{eje}_km" for eje in "xyz"]].to_numpy()
posiciones_sol_km = tabla[[f"sol_{eje}_km" for eje in "xyz"]].to_numpy()
iluminacion = tabla["iluminado_sin_terreno"].to_numpy(dtype=bool)
conexion_tierra = tabla["conexion_tierra"].to_numpy(dtype=bool)

# Resumen del primer instante, obtenido de la misma tabla para ambos cuerpos.
primera_fila = tabla.iloc[0]
resumen = pd.DataFrame([
    {"cuerpo": cuerpo.title(), **{
        campo: primera_fila[columna.format(cuerpo=cuerpo)]
        for campo, columna in CAMPOS_RESUMEN.items()
    }}
    for cuerpo in CUERPOS
])

print("Lugar:", tabla.attrs["lugar"])
print(f"Muestras: {len(tabla)} | Desde {tabla['tiempo_utc'].iloc[0]} hasta {tabla['tiempo_utc'].iloc[-1]}")
display(tabla_terreno.head(FILAS_VISTA))
display(resumen.round(3))

if MOSTRAR_CABECERAS:
    print(tabla.attrs["cabeceras_horizons"]["Tierra"])


Lugar: {'nombre': 'Apolo 11', 'latitud_deg': 0.67416, 'longitud_deg': 23.47314, 'altura_m': 0.0}
Muestras: 1 | Desde 1969-07-20 20:17:40+00:00 hasta 1969-07-20 20:17:40+00:00


,tiempo_utc,t_segundos,elevacion_tierra_deg,acimut_tierra_deg,angulo_normal_tierra_deg,distancia_tierra_km,elevacion_sol_deg,acimut_sol_deg,angulo_normal_sol_deg,distancia_sol_km,sol_sobre_horizonte,tierra_bloquea_sol,iluminado_sin_terreno,tierra_visible,conexion_tierra
0,1969-07-20 20:17:40+00:00,0.0,59.224427,272.195293,30.775573,387974.593565,10.697766,88.805517,79.302234,1.518713e+08,True,False,True,True,True


,cuerpo,x_km,y_km,z_km,distancia_km,elevacion_deg,acimut_deg,angulo_normal_deg
0,Tierra,3.839234e+05,4.690012e+04,3.045522e+04,3.879746e+05,59.224,272.195,30.776
1,Sol,-7.177629e+07,1.227884e+08,5.325471e+07,1.518713e+08,10.698,88.806,79.302


## Significado de los resultados

- `tabla`: UTC, segundos desde el inicio, posiciones XYZ, distancias, ángulos y booleanos.
- `tabla_terreno`: tiempo, geometría y filtros para el modelo topográfico.
- `resumen`: Tierra y Sol en el primer instante.
- `datos`: array con las columnas de `COLUMNAS_ARRAY`: tiempo en segundos, XYZ de Tierra y XYZ de Sol.
- `datos_geometria`: tiempo en segundos seguido de `COLUMNAS_GEOMETRIA`.
- `tiempos_utc`, `posiciones_tierra_km`, `posiciones_sol_km`, `iluminacion` y `conexion_tierra`: arrays separados.

Las posiciones parten del lugar configurado y apuntan al centro de cada cuerpo, en **km y ejes ICRF**.
La normal es radial y el norte pertenece al modelo lunar Mean Earth de Horizons.

| Campo para cada cuerpo | Definición |
| --- | --- |
| `distancia_*_km` | Distancia desde el lugar al centro del cuerpo. |
| `elevacion_*_deg` | Ángulo sobre el horizontal local: de -90° a +90°. |
| `acimut_*_deg` | Norte lunar 0°, este 90°, sur 180°, oeste 270°. |
| `angulo_normal_*_deg` | Theta respecto a la normal exterior: 90° menos la elevación. |

El acimut es `NaN` en los polos o en cénit/nadir, donde no está definido.

| Booleano | Condición |
| --- | --- |
| `sol_sobre_horizonte` | Centro del Sol por encima del horizontal local. |
| `tierra_bloquea_sol` | El segmento hacia el centro solar intersecta la Tierra esférica. |
| `iluminado_sin_terreno` | Sol sobre el horizonte y Tierra sin bloquearlo. |
| `iluminado` | Alias del resultado anterior. |
| `tierra_visible` / `conexion_tierra` | Centro terrestre con elevación mayor que cero. |

**Modelo:** Sol puntual, Luna esférica y Tierra de radio 6371 km, sin relieve,
atmósfera, eclipses parciales ni correcciones de luz/aberración. El horizonte exacto se
considera no visible. A altitud cero, el horizonte también descarta que la Luna bloquee el cuerpo.
La altura modifica los vectores y ángulos, pero el filtro no incluye la depresión del horizonte
por altura. La conexión es visibilidad de la Tierra, sin evaluar una estación o enlace radio.

La altura se mide sobre la superficie de referencia de Horizons, no sobre el terreno local.
El valor cero del ejemplo no es una elevación medida del Apolo 11. La topografía debe añadir
la orientación de la pendiente y el horizonte del relieve según el acimut de cada cuerpo.

`DURACION=0` consulta un instante. Para una serie, usa duración positiva y un paso de al menos
un minuto entero. Si el final coincide con un paso se incluye; si no, la última muestra queda
antes del final. Se permiten hasta 10000 muestras. Las fechas sin zona se interpretan como UTC,
desde 1962; las que tienen zona se convierten a UTC.

La función interna `posiciones_desde_luna(..., sitio_lunar=None)` sigue disponible para
consultas con origen en el centro lunar, sin ángulos locales ni booleanos de superficie.

Exportación opcional, sin repetir consultas:

```python
tabla.to_csv("geometria_lunar.csv", index=False)
np.save("posiciones_luna.npy", datos)
```

Referencias: [JPL Horizons API](https://ssd-api.jpl.nasa.gov/doc/horizons.html),
[manual de marcos, tiempos y coordenadas](https://ssd.jpl.nasa.gov/horizons/manual.html),
[radio terrestre medio](https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html).


In [1]:
from procesar_sitio_lunar import procesar_peticion_frontend

resultado = procesar_peticion_frontend(
    tiempo="1969-7-20T20:17:00Z",
    latitud=-0.674,
    longitud=23.473,
)
resultado

HTTPError: 404 Client Error: Not Found for url: https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/data/lola_gdr/cylindrical/img/ldem_1024_15s_00n_000_030.lbl